# Reverse Model Text-Only Inversion

In [ ]:
# For when connected to google colab
!git clone https://github.com/AdrSkapars/inversion_optimisation.git 
%cd inversion_optimisation
!uv pip install -e .

## Set Up Model

In [1]:
import sys
sys.path.insert(0, "/content/inversion_optimisation/src")

import torch
from datasets import load_dataset
from transformer_lens import HookedTransformer
import numpy as np
import random
import pickle
from tqdm import tqdm
import json
import logging
import time

from inversion_optimisation.utils import (
    get_paper_summary_stats_new, 
    DotDict,
    DATA_PATH,
)

/workspace/inversion_optimisation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the LLM model

device = ["cuda", "cpu"][0]
model_name = {
    # Simple models
    0: "attn-only-1l",
    1: "gelu-1l",
    2: "tiny-stories-1L-21M",
    
    ## Small models
    3: "tiny-stories-1M",
    4: "tiny-stories-3M",
    5: "tiny-stories-8M",
    6: "tiny-stories-28M",
    7: "tiny-stories-33M",
    8: "tiny-stories-instruct-33M",
    
    ## Large models
    9: "gpt2-small",
    10: "gpt2-medium",
    11: "gpt2-xl",
    12: "llama-7b",
    13: "meta-llama/Llama-3.2-1B",
    14: "meta-llama/Llama-3.2-1B-Instruct",
    15: "Qwen/Qwen2.5-0.5B",
    16: "Qwen/Qwen2.5-1.5B",
    17: "Qwen/Qwen2.5-3B",
    18: "pythia-1.4b",
}[7]

model = HookedTransformer.from_pretrained(model_name)#, device=device)
model = model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model tiny-stories-33M into HookedTransformer


## Finetune Reverse Model

In [16]:
from inversion_optimisation.algorithms.reverse_model_lora import (
    generate_training_pairs,
    train_reverse_model_lora,
)
from inversion_optimisation.algorithms.reverse_model import reverse_model_text_search
from inversion_optimisation.inv_configs import REVERSE_MODEL_LORA_TEXT_CFG

cfg = REVERSE_MODEL_LORA_TEXT_CFG

In [4]:
dataset_name = "tinystories"  # change to "tinystories" or "reddit"

# Generate training pairs (uses the forward TinyStories model)
text_pairs = generate_training_pairs(
    model, dataset_name,
    num_examples=cfg.num_train_examples,
    min_input_len=cfg.min_input_len,
    max_input_len=cfg.max_input_len,
    num_output_tokens=cfg.num_output_tokens,
    device=device,
    seed=cfg.train_seed,
)
print(f"Generated {len(text_pairs)} training pairs")
print(f"Example: input={text_pairs[0][0]!r}  output={text_pairs[0][1]!r}")


Loading tinystories data: 101378it [00:44, 2297.31it/s]
Generating outputs: 100%|██████████| 10/10 [01:12<00:00,  7.26s/it]


Generated 100000 training pairs
Example: input=' named'  output=' Lily. She loved to play outside in the sunshine. One day, she saw a big, red apple on the ground.'


In [5]:
cfg

{'reverse_model_id': 'Corning/Reverse-Model-7B-348B',
 'lora_rank': 32,
 'lora_alpha': 64,
 'lora_dropout': 0.05,
 'num_epochs': 2,
 'batch_size': 8,
 'gradient_accumulation_steps': 4,
 'learning_rate': 0.0002,
 'warmup_ratio': 0.05,
 'max_length': 128,
 'val_split': 0.05,
 'train_seed': 42,
 'num_train_examples': 100000,
 'min_input_len': 1,
 'max_input_len': 10,
 'num_output_tokens': 25,
 'save_folder': 'ReverseModelLoRA_TinyStories33M_text',
 'model_name': 'tiny-stories-33M'}

In [ ]:
# Do hyperparam experiment

for lr in [0.005, 0.002, 0.0005, 0.0001]:
    print(f"\n{'='*40} LR={lr} {'='*40}")
    lora_model, lora_tokenizer = train_reverse_model_lora(
        text_pairs,
        model_id=cfg.reverse_model_id,
        output_dir=f"data/ReverseModelLoRA_{dataset_name}_lr{lr}",
        lora_rank=cfg.lora_rank,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        num_epochs=1,
        batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=lr,
        warmup_ratio=0.01,
        max_length=cfg.max_length,
        val_split=cfg.val_split,
        seed=cfg.train_seed,
        logging_steps=10,
        max_steps=500,    # ~0.1 epoch, stops early
    )
    del lora_model  # free GPU memory
    torch.cuda.empty_cache()


In [10]:
output_dir = f"data/ReverseModelLoRA_{dataset_name}"

lora_model, lora_tokenizer = train_reverse_model_lora(
    text_pairs,
    model_id=cfg.reverse_model_id,
    output_dir=output_dir,
    lora_rank=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    num_epochs=4,
    batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=0.0005,
    warmup_ratio=cfg.warmup_ratio,
    max_length=cfg.max_length,
    val_split=cfg.val_split,
    seed=cfg.train_seed,
)


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 22,020,096 || all params: 7,815,736,320 || trainable%: 0.2817


Epoch,Training Loss,Validation Loss
1,1.893700,1.895615
2,1.789000,1.802098
3,1.565600,1.734709
4,1.298600,1.737705


In [14]:
from huggingface_hub import login

repo_id = f"AdrSkapars/reverse-model-lora-{dataset_name}"
lora_path = f"data/ReverseModelLoRA_{dataset_name}/best_lora"

lora_model.push_to_hub(repo_id)
lora_tokenizer.push_to_hub(repo_id)
print(f"Uploaded to https://huggingface.co/{repo_id}")

Processing Files (1 / 1): 100%|██████████| 88.1MB / 88.1MB, 29.5MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.
Processing Files (1 / 1): 100%|██████████| 1.00MB / 1.00MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded to https://huggingface.co/AdrSkapars/reverse-model-lora-tinystories


# ------------------------

## Reverse Model Experiments (Off-shelf)

In [3]:
from inversion_optimisation.algorithms.reverse_model import reverse_model_text_search
from inversion_optimisation.inv_configs import REVERSE_MODEL_TEXT_CFG

cfg = REVERSE_MODEL_TEXT_CFG

In [ ]:
# Set parameters for dataset size
cfg.input_len = 1
cfg.output_len = 25
cfg.num_targets = 1000
cfg.target_strategy = ["random", "reddit", "tinystories"][0]

# Set random seeds
torch.manual_seed(1)
np.random.seed(1)
random.seed(1)

# Generate the targets used for all experiments
tokens_list = []
for _ in tqdm(range(cfg.num_targets)):
    tokens = torch.randint(0, len(model.tokenizer.vocab), (1, cfg.input_len)).to(device)
    tokens_list.append(tokens)
true_tokens = torch.cat(tokens_list, dim=0).to(device)
with open(DATA_PATH / f'true_tokens_{cfg.num_targets}_{cfg.input_len}.pkl', 'wb') as file:
    pickle.dump(true_tokens, file)

batch_size = 1000
for batch in range(0, cfg.num_targets, batch_size):
    input_tokens = true_tokens[batch:batch+batch_size].to(device)
    output_tokens = model.generate(
        input_tokens,
        max_new_tokens=cfg.output_len,
        do_sample=False,
        stop_at_eos=False,
        verbose=False,
        return_type="tokens",)[:,cfg.input_len:]
    if batch == 0:
        all_output_tokens = output_tokens
    else:
        all_output_tokens = torch.cat((all_output_tokens, output_tokens), dim=0)
with open(DATA_PATH / f"true_tokens_{cfg.num_targets}_{cfg.input_len}_{cfg.output_len}_greedy.pkl", 'wb') as file:
    pickle.dump(all_output_tokens, file)

results, elapsed_time = reverse_model_text_search(cfg, model, device)
print()
stats = get_paper_summary_stats_new(results, epochs=1)
stats["elapsed_time"] = elapsed_time
stats["experiment_params"] = cfg
print("input_len", cfg.input_len, "\tpercent_zero_loss", stats["percent_zero_loss"], "\tpercent_exact_inversion", stats["percent_exact_inversion"], "\telapsed_time", stats["elapsed_time"])

for i in range(10):
    true_text = model.tokenizer.decode(results[i]["true_tokens"].tolist())
    pred_text = model.tokenizer.decode(results[i]["pred_tokens"].tolist())
    match = "✓" if results[i]["found_solution"] else "✗"
    print(f"{i}: true={[true_text]}  pred={[pred_text]}  {match}")


# with open(DATA_PATH / f'{cfg.save_folder}/stats_{cfg.input_len}_{cfg.num_targets}.json', 'w') as f:
#     json.dump(stats, f)
# with open(DATA_PATH / f'{cfg.save_folder}/results_{cfg.input_len}_{cfg.num_targets}.pkl', 'wb') as f:
#     pickle.dump(results, f)

In [10]:
for target_strategy in ["tinystories"]:
    for input_len in [1,2,3,4,5]:

        # Set parameters for dataset size
        cfg.input_len = input_len #1
        cfg.output_len = 25
        cfg.num_targets = 1000
        cfg.target_strategy = target_strategy #["random", "reddit", "tinystories"][0]

        # Set random seeds
        torch.manual_seed(1)
        np.random.seed(1)
        random.seed(1)

        # Generate the targets used for all experiments
        tokens_list = []
        for _ in tqdm(range(cfg.num_targets)):
            tokens = torch.randint(0, len(model.tokenizer.vocab), (1, cfg.input_len)).to(device)
            tokens_list.append(tokens)
        true_tokens = torch.cat(tokens_list, dim=0).to(device)
        with open(DATA_PATH / f'true_tokens_{cfg.num_targets}_{cfg.input_len}.pkl', 'wb') as file:
            pickle.dump(true_tokens, file)

        batch_size = 1000
        for batch in range(0, cfg.num_targets, batch_size):
            input_tokens = true_tokens[batch:batch+batch_size].to(device)
            output_tokens = model.generate(
                input_tokens,
                max_new_tokens=cfg.output_len,
                do_sample=False,
                stop_at_eos=False,
                verbose=False,
                return_type="tokens",)[:,cfg.input_len:]
            if batch == 0:
                all_output_tokens = output_tokens
            else:
                all_output_tokens = torch.cat((all_output_tokens, output_tokens), dim=0)
        with open(DATA_PATH / f"true_tokens_{cfg.num_targets}_{cfg.input_len}_{cfg.output_len}_greedy.pkl", 'wb') as file:
            pickle.dump(all_output_tokens, file)

        results, elapsed_time = reverse_model_text_search(cfg, model, device)
        print()
        stats = get_paper_summary_stats_new(results, epochs=1)
        stats["elapsed_time"] = elapsed_time
        stats["experiment_params"] = cfg
        print("input_len", cfg.input_len, "\tpercent_zero_loss", stats["percent_zero_loss"], "\tpercent_exact_inversion", stats["percent_exact_inversion"], "\telapsed_time", stats["elapsed_time"])

        for i in range(10):
            true_text = model.tokenizer.decode(results[i]["true_tokens"].tolist())
            pred_text = model.tokenizer.decode(results[i]["pred_tokens"].tolist())
            match = "✓" if results[i]["found_solution"] else "✗"
            print(f"{i}: true={[true_text]}  pred={[pred_text]}  {match}")

        with open(DATA_PATH / f'{cfg.save_folder}/stats_{cfg.input_len}_{cfg.num_targets}.json', 'w') as f:
            json.dump(stats, f)
        with open(DATA_PATH / f'{cfg.save_folder}/results_{cfg.input_len}_{cfg.num_targets}.pkl', 'wb') as f:
            pickle.dump(results, f)

  0%|          | 0/1000 [00:00<?, ?it/s]

Reverse model inversion: 100%|██████████| 1000/1000 [02:03<00:00,  8.08it/s]


100%|██████████| 1/1 [00:00<00:00, 172.28it/s]


input_len 1 	percent_zero_loss 0.2 	percent_exact_inversion 0.2 	elapsed_time 123.8189971446991
0: true=['Spot']  pred=['name']  ✗
1: true=['R']  pred=[',']  ✗
2: true=['And']  pred=['t']  ✗
3: true=['S']  pred=['there']  ✗
4: true=['One']  pred=['t']  ✗
5: true=['One']  pred=['in']  ✗
6: true=['He']  pred=['there']  ✗
7: true=['He']  pred=['name']  ✗
8: true=['He']  pred=['there']  ✗
9: true=['M']  pred=['was']  ✗


Reverse model inversion: 100%|██████████| 1000/1000 [03:54<00:00,  4.26it/s]


...

100%|██████████| 4/4 [00:00<00:00, 196.52it/s]


input_len 4 	percent_zero_loss 0.0 	percent_exact_inversion 0.0 	elapsed_time 457.3967525959015
0: true=['He was very sad']  pred=['takes all the']  ✗
1: true=['He loved to run']  pred=['". He didn\'t']  ✗
2: true=['The old man continued']  pred=['It was wonderful to']  ✗
3: true=['He was very tall']  pred=['how long it was']  ✗
4: true=['From that day on']  pred=["she couldn't do"]  ✗
5: true=['The next day,']  pred=[', but she was']  ✗
6: true=['He called her over']  pred=['in your beard is']  ✗
7: true=['And when the fire']  pred=['the world. Ever']  ✗
8: true=['Max and Kiki']  pred=['it, but he']  ✗
9: true=['Josh was so happy']  pred=['.\n\nNow']  ✗


Reverse model inversion: 100%|██████████| 1000/1000 [09:32<00:00,  1.75it/s]


100%|██████████| 5/5 [00:00<00:00, 198.49it/s]


input_len 5 	percent_zero_loss 0.0 	percent_exact_inversion 0.0 	elapsed_time 572.0947895050049
0: true=['You have your own cr']  pred=[', then you are not']  ✗
1: true=['Tom and Lily took some']  pred=['ried and she cried until']  ✗
2: true=['One day, they see']  pred=['was just the two of']  ✗
3: true=['Tom sticks his head into']  pred=['"Huh? What did']  ✗
4: true=['They also had a spoon']  pred=['fell to the ground.']  ✗
5: true=['I made it for you']  pred=["You can't do this"]  ✗
6: true=['They want to go to']  pred=['upon a time, there']  ✗
7: true=['"The duckling belongs']  pred=['at me, and I']  ✗
8: true=['They see a man with']  pred=['. It was proud of']  ✗
9: true=['You can gain something better']  pred=['\nThere was supposed to']  ✗


## Reverse Model Experiments (Finetuned)

In [21]:
for input_len in [1,2,3,4,5]:

    # Set eval config
    eval_cfg = DotDict(REVERSE_MODEL_LORA_TEXT_CFG.copy())
    eval_cfg.input_len = input_len
    eval_cfg.output_len = 25
    eval_cfg.num_targets = 1000
    eval_cfg.target_strategy = dataset_name  # match training distribution
    eval_cfg.lora_path = f"data/ReverseModelLoRA_{dataset_name}/best_lora"

    # Set random seeds
    torch.manual_seed(1)
    np.random.seed(1)
    random.seed(1)

    # Generate eval targets (same as other methods)
    tokens_list = []
    for _ in tqdm(range(eval_cfg.num_targets)):
        tokens = torch.randint(0, len(model.tokenizer.vocab), (1, eval_cfg.input_len)).to(device)
        tokens_list.append(tokens)
    true_tokens = torch.cat(tokens_list, dim=0).to(device)
    with open(DATA_PATH / f'true_tokens_{eval_cfg.num_targets}_{eval_cfg.input_len}.pkl', 'wb') as file:
        pickle.dump(true_tokens, file)

    batch_size = 1000
    for batch in range(0, eval_cfg.num_targets, batch_size):
        input_tokens = true_tokens[batch:batch+batch_size].to(device)
        output_tokens = model.generate(
            input_tokens, max_new_tokens=eval_cfg.output_len,
            do_sample=False, stop_at_eos=False, verbose=False, return_type="tokens",
        )[:,eval_cfg.input_len:]
        if batch == 0:
            all_output_tokens = output_tokens
        else:
            all_output_tokens = torch.cat((all_output_tokens, output_tokens), dim=0)
    with open(DATA_PATH / f"true_tokens_{eval_cfg.num_targets}_{eval_cfg.input_len}_{eval_cfg.output_len}_greedy.pkl", 'wb') as file:
        pickle.dump(all_output_tokens, file)

    results, elapsed_time = reverse_model_text_search(eval_cfg, model, device)
    print()
    stats = get_paper_summary_stats_new(results, epochs=1)
    stats["elapsed_time"] = elapsed_time
    print("input_len", eval_cfg.input_len, "\tpercent_exact_inversion", stats["percent_exact_inversion"], "\telapsed_time", stats["elapsed_time"])

    for i in range(10):
        true_text = model.tokenizer.decode(results[i]["true_tokens"].tolist())
        pred_text = model.tokenizer.decode(results[i]["pred_tokens"].tolist())
        match = "✓" if results[i]["found_solution"] else "✗"
        print(f"{i}: true={[true_text]}  pred={[pred_text]}  {match}")

    with open(DATA_PATH / f'{eval_cfg.save_folder}/stats_{eval_cfg.input_len}_{eval_cfg.num_targets}.json', 'w') as f:
        json.dump(stats, f)
    with open(DATA_PATH / f'{eval_cfg.save_folder}/results_{eval_cfg.input_len}_{eval_cfg.num_targets}.pkl', 'wb') as f:
        pickle.dump(results, f)

  0%|          | 0/1000 [00:00<?, ?it/s]

Reverse model inversion: 100%|██████████| 1000/1000 [03:10<00:00,  5.26it/s]


100%|██████████| 1/1 [00:00<00:00, 181.40it/s]


input_len 1 	percent_exact_inversion 0.1 	elapsed_time 190.29499745368958
0: true=['Spot']  pred=['it']  ✗
1: true=['R']  pred=['and']  ✗
2: true=['And']  pred=['he']  ✗
3: true=['S']  pred=['it']  ✗
4: true=['One']  pred=['it']  ✗
5: true=['One']  pred=['she']  ✗
6: true=['He']  pred=['it']  ✗
7: true=['He']  pred=['it']  ✗
8: true=['He']  pred=['it']  ✗
9: true=['M']  pred=['was']  ✗


Reverse model inversion: 100%|██████████| 1000/1000 [06:12<00:00,  2.69it/s]


...

100%|██████████| 4/4 [00:00<00:00, 196.54it/s]


input_len 4 	percent_exact_inversion 0.0 	elapsed_time 735.6227488517761
0: true=['He was very sad']  pred=['the end, they']  ✗
1: true=['He loved to run']  pred=[', but he was']  ✗
2: true=['The old man continued']  pred=['of the day,']  ✗
3: true=['He was very tall']  pred=['"You did a']  ✗
4: true=['From that day on']  pred=['Bonnie tried and']  ✗
5: true=['The next day,']  pred=['time there was a']  ✗
6: true=['He called her over']  pred=['€�Iâ']  ✗
7: true=['And when the fire']  pred=['day he met a']  ✗
8: true=['Max and Kiki']  pred=['himself, so']  ✗
9: true=['Josh was so happy']  pred=['you play baseball?"']  ✗


Reverse model inversion: 100%|██████████| 1000/1000 [15:11<00:00,  1.10it/s]


100%|██████████| 5/5 [00:00<00:00, 191.05it/s]


input_len 5 	percent_exact_inversion 0.0 	elapsed_time 911.1633615493774
0: true=['You have your own cr']  pred=[', and in the end']  ✗
1: true=['Tom and Lily took some']  pred=['yummy!" Lily\'s']  ✗
2: true=['One day, they see']  pred=['it was hard at first']  ✗
3: true=['Tom sticks his head into']  pred=['ook her head and thought']  ✗
4: true=['They also had a spoon']  pred=['" The children danced and']  ✗
5: true=['I made it for you']  pred=['boy was so excited that']  ✗
6: true=['They want to go to']  pred=['Alfred worked hard']  ✗
7: true=['"The duckling belongs']  pred=['Fan was worried and said']  ✗
8: true=['They see a man with']  pred=['to climb the tree,']  ✗
9: true=['You can gain something better']  pred=['little boy was so proud']  ✗


In [19]:
# Quick comparison on a few examples
from inversion_optimisation.algorithms.reverse_model import load_reverse_model, generate_reverse

rev_base, tok = load_reverse_model()
rev_lora, tok = load_reverse_model(lora_path=f"data/ReverseModelLoRA_{dataset_name}/best_lora")

test_output = model.tokenizer.decode(loaded_true_outputs[0].tolist())
print("Base:", generate_reverse(rev_base, tok, test_output, max_new_tokens=10))
print("LoRA:", generate_reverse(rev_lora, tok, test_output, max_new_tokens=10))


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.18it/s]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Base: there was a box with his name on it. He was so excited to see what was inside.

When he opened the box, he found a big, shiny
LoRA: was very curious, so he decided to open it He was so excited to see what was inside.

When he opened the box, he found a big, shiny


In [20]:
import pickle
from inversion_optimisation.utils import DATA_PATH

with open(DATA_PATH / "true_tokens_1000_1_25_greedy.pkl", 'rb') as f:
    loaded_true_outputs = pickle.load(f)

test_output = model.tokenizer.decode(loaded_true_outputs[0].tolist())
print("Base:", generate_reverse(rev_base, tok, test_output, max_new_tokens=10))
print("LoRA:", generate_reverse(rev_lora, tok, test_output, max_new_tokens=10))


Base: there was a box with his name on it. He was so excited to see what was inside.

When he opened the box, he found a big, shiny
LoRA: was very curious, so he decided to open it He was so excited to see what was inside.

When he opened the box, he found a big, shiny
